# Agent-Based SIR Model
## Lesson 3 · Section 2

In this notebook you will build a simple **agent-based epidemic model** step by step.

We will:
1. Define the health states of agents
2. Implement movement on a square grid
3. Test infection and recovery rules on small examples
4. Simulate a full population over time
5. Visualise both spatial spread and SIR time series
6. Explore how randomness changes the outcome
7. End with suggestions for independent work

The notebook is intentionally split into small, testable parts so you can verify each rule before using it in a full simulation.


## 0 · Imports
We use NumPy, Matplotlib, and Python's random module.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('Imports loaded successfully ✓')


---
## 1 · Agent states
Each agent represents one individual. Each individual can be in one of three states:

- `1` = susceptible
- `2` = infected
- `3` = recovered


In [ ]:
STATE_SUSCEPTIBLE = 1
STATE_INFECTED = 2
STATE_RECOVERED = 3

state_names = {
    STATE_SUSCEPTIBLE: 'Susceptible',
    STATE_INFECTED: 'Infected',
    STATE_RECOVERED: 'Recovered',
}

print(state_names)


## 2 · Movement rule
We start with the same simple movement idea as in the lesson code: during each step, an agent moves by one square in one random direction, while staying inside the simulation area.


In [ ]:
def update_position(x, y, area, rng=random):
    direction = rng.randint(1, 4)
    if direction == 1:
        if y < area:
            y += 1
    elif direction == 2:
        if y > 0:
            y -= 1
    elif direction == 3:
        if x > 0:
            x -= 1
    else:
        if x < area:
            x += 1
    return x, y

print(update_position(5, 5, area=10))


### Independent test 1
Test the movement rule at the boundary. The position should remain inside the area.


In [ ]:
random.seed(1)
for _ in range(10):
    x_new, y_new = update_position(0, 0, area=5)
    assert 0 <= x_new <= 5
    assert 0 <= y_new <= 5
print('Test passed ✓ Boundary handling works.')


---
## 3 · Detect whether infection is nearby
A susceptible agent becomes at risk if an infected agent is in the same grid square.

This is a simple contact rule. Later you can make it more realistic.


In [ ]:
def is_infected_nearby(person_index, positions, states):
    same_x = positions[:, 0] == positions[person_index, 0]
    same_y = positions[:, 1] == positions[person_index, 1]
    infected = states == STATE_INFECTED
    not_same_person = np.arange(len(states)) != person_index
    return np.any(same_x & same_y & infected & not_same_person)


### Independent test 2
Use a tiny hand-made example so the infection rule is easy to verify.


In [ ]:
test_positions = np.array([
    [1, 1],
    [2, 2],
    [1, 1],
    [3, 3],
])

test_states = np.array([
    STATE_SUSCEPTIBLE,
    STATE_INFECTED,
    STATE_INFECTED,
    STATE_RECOVERED,
])

person = 0
result = is_infected_nearby(person, test_positions, test_states)
print(f'Is infected agent nearby person {person}? {result}')
assert bool(result)
print('Test passed ✓ The susceptible agent shares a cell with an infected one.')

---
## 4 · Update the health state of one agent
We now write one function that applies the epidemic rules:

- susceptible + infected nearby → may become infected
- infected → may recover
- recovered → stays recovered


In [ ]:
def update_health_state(current_state, infected_nearby, infection_probability, recovery_probability, rng=random):
    if current_state == STATE_SUSCEPTIBLE:
        if infected_nearby and rng.random() < infection_probability:
            return STATE_INFECTED
        return STATE_SUSCEPTIBLE

    if current_state == STATE_INFECTED:
        if rng.random() < recovery_probability:
            return STATE_RECOVERED
        return STATE_INFECTED

    return STATE_RECOVERED


### Independent test 3
Force infection and recovery with probability `1.0` so the expected result is obvious.


In [ ]:
random.seed(2)
result_1 = update_health_state(STATE_SUSCEPTIBLE, True, infection_probability=1.0, recovery_probability=0.0)
result_2 = update_health_state(STATE_INFECTED, False, infection_probability=0.0, recovery_probability=1.0)
print('Susceptible exposed with p=1.0 ->', result_1)
print('Infected with recovery p=1.0 ->', result_2)
assert result_1 == STATE_INFECTED
assert result_2 == STATE_RECOVERED
print('Test passed ✓ Health-state update works for simple deterministic cases.')


---
## 5 · Simulate one full time step for all agents
To avoid order effects, we first compute the new health states and only then replace the old array.


In [ ]:
def step_agent_sir(positions, states, area, infection_probability, recovery_probability, rng=random):
    new_states = states.copy()

    for person in range(len(states)):
        infected_nearby = is_infected_nearby(person, positions, states)
        new_states[person] = update_health_state(
            states[person], infected_nearby, infection_probability, recovery_probability, rng=rng
        )

    new_positions = positions.copy()
    for person in range(len(states)):
        x, y = update_position(int(positions[person, 0]), int(positions[person, 1]), area, rng=rng)
        new_positions[person] = [x, y]

    return new_positions, new_states


### Independent test 4
Create a tiny example where one susceptible agent must become infected.


In [ ]:
random.seed(3)
small_positions = np.array([
    [1, 1],
    [1, 1],
    [4, 4],
])
small_states = np.array([
    STATE_SUSCEPTIBLE,
    STATE_INFECTED,
    STATE_RECOVERED,
])

_, updated_states = step_agent_sir(
    small_positions,
    small_states,
    area=5,
    infection_probability=1.0,
    recovery_probability=0.0,
)

print('Updated states:', updated_states)
assert updated_states[0] == STATE_INFECTED
print('Test passed ✓ Local infection spreads as expected.')


---
## 6 · Full agent-based simulation
Now we combine all previous pieces into one complete model.


In [ ]:
def simulate_agent_sir(
    area=40,
    population=200,
    initially_infected=5,
    steps=150,
    infection_probability=0.25,
    recovery_probability=0.03,
    seed=1,
):
    rng = random.Random(seed)
    positions = np.array(
        [[rng.randint(0, area), rng.randint(0, area)] for _ in range(population)],
        dtype=int,
    )
    states = np.full(population, STATE_SUSCEPTIBLE, dtype=int)
    states[:initially_infected] = STATE_INFECTED

    susceptible_count = np.zeros(steps, dtype=int)
    infected_count = np.zeros(steps, dtype=int)
    recovered_count = np.zeros(steps, dtype=int)
    saved_positions = []
    saved_states = []

    for step in range(steps):
        saved_positions.append(positions.copy())
        saved_states.append(states.copy())
        susceptible_count[step] = np.sum(states == STATE_SUSCEPTIBLE)
        infected_count[step] = np.sum(states == STATE_INFECTED)
        recovered_count[step] = np.sum(states == STATE_RECOVERED)
        positions, states = step_agent_sir(
            positions,
            states,
            area=area,
            infection_probability=infection_probability,
            recovery_probability=recovery_probability,
            rng=rng,
        )

    return {
        'positions': saved_positions,
        'states': saved_states,
        'susceptible': susceptible_count,
        'infected': infected_count,
        'recovered': recovered_count,
        'final_states': states,
        'area': area,
    }

result = simulate_agent_sir()
print('Simulation finished ✓')
print('Peak infected:', result['infected'].max())

### Independent test 5
The counts of susceptible, infected, and recovered agents should always sum to the total population.


In [ ]:
population = result['susceptible'][0] + result['infected'][0] + result['recovered'][0]
total = result['susceptible'] + result['infected'] + result['recovered']
print('First five totals:', total[:5])
assert np.all(total == population)
print('Test passed ✓ Agent counts are conserved.')


---
## 7 · Visualise one run
We show:
- one spatial snapshot of the agents
- the time series of `S`, `I`, and `R`


In [ ]:
def plot_snapshot_and_curves(simulation_result, snapshot_step=30):
    positions = simulation_result['positions'][snapshot_step]
    states = simulation_result['states'][snapshot_step]
    S = simulation_result['susceptible']
    I = simulation_result['infected']
    R = simulation_result['recovered']
    area = simulation_result['area']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(positions[states == STATE_SUSCEPTIBLE, 0], positions[states == STATE_SUSCEPTIBLE, 1], c='green', s=20, label='S')
    axes[0].scatter(positions[states == STATE_INFECTED, 0], positions[states == STATE_INFECTED, 1], c='red', s=20, label='I')
    axes[0].scatter(positions[states == STATE_RECOVERED, 0], positions[states == STATE_RECOVERED, 1], c='blue', s=20, label='R')
    axes[0].set_xlim(0, area)
    axes[0].set_ylim(0, area)
    axes[0].set_aspect('equal')
    axes[0].set_title(f'Spatial snapshot at step {snapshot_step}')
    axes[0].legend()

    axes[1].plot(S, color='green', linewidth=2, label='S')
    axes[1].plot(I, color='red', linewidth=2, label='I')
    axes[1].plot(R, color='blue', linewidth=2, label='R')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Number of agents')
    axes[1].set_title('Agent-based SIR time series')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_snapshot_and_curves(result, snapshot_step=30)

---
## 8b · Multiple Snapshots Over Time

Let's visualise the spatial spread of the epidemic at several time points.

In [ ]:
def plot_spatial_snapshots(simulation_result, snapshot_steps=[0, 20, 50, 100]):
    """Show the spatial state of agents at multiple time points."""
    n = len(snapshot_steps)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    area = simulation_result['area']
    
    for ax, step in zip(axes, snapshot_steps):
        pos = simulation_result['positions'][min(step, len(simulation_result['positions'])-1)]
        st = simulation_result['states'][min(step, len(simulation_result['states'])-1)]
        ax.scatter(pos[st == STATE_SUSCEPTIBLE, 0], pos[st == STATE_SUSCEPTIBLE, 1],
                   c='green', s=10, alpha=0.6)
        ax.scatter(pos[st == STATE_INFECTED, 0], pos[st == STATE_INFECTED, 1],
                   c='red', s=15, zorder=5)
        ax.scatter(pos[st == STATE_RECOVERED, 0], pos[st == STATE_RECOVERED, 1],
                   c='blue', s=10, alpha=0.6)
        ax.set_xlim(0, area)
        ax.set_ylim(0, area)
        ax.set_aspect('equal')
        n_s = np.sum(st == STATE_SUSCEPTIBLE)
        n_i = np.sum(st == STATE_INFECTED)
        n_r = np.sum(st == STATE_RECOVERED)
        ax.set_title(f't = {step}\nS={n_s} I={n_i} R={n_r}', fontsize=10)
    
    plt.suptitle('Spatial Spread of Epidemic Over Time', fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_spatial_snapshots(result, snapshot_steps=[0, 30, 60, 120])

---
## 8c · Distance-Based Infection Rule

The original model requires agents to share the **exact same cell**. A more realistic rule: infection occurs if an infected agent is within a certain **distance**.

In [ ]:
def is_infected_nearby_distance(person_index, positions, states, radius=2.0):
    """Check if any infected agent is within 'radius' of the given person."""
    dx = positions[:, 0] - positions[person_index, 0]
    dy = positions[:, 1] - positions[person_index, 1]
    dist = np.sqrt(dx**2 + dy**2)
    infected = states == STATE_INFECTED
    not_self = np.arange(len(states)) != person_index
    return np.any((dist <= radius) & infected & not_self)

# Test: two agents within radius
test_pos = np.array([[5, 5], [6, 5], [20, 20]])
test_st = np.array([STATE_SUSCEPTIBLE, STATE_INFECTED, STATE_RECOVERED])
assert is_infected_nearby_distance(0, test_pos, test_st, radius=2.0)
assert not is_infected_nearby_distance(2, test_pos, test_st, radius=2.0)
print('Test passed: distance-based infection detection works.')

# Compare same-cell vs distance-based
def simulate_abm_sir_distance(area=40, population=200, initially_infected=5,
                               steps=150, infection_prob=0.15, recovery_prob=0.03,
                               radius=2.0, seed=1):
    rng = random.Random(seed)
    positions = np.array([[rng.randint(0, area), rng.randint(0, area)]
                          for _ in range(population)], dtype=int)
    states = np.full(population, STATE_SUSCEPTIBLE, dtype=int)
    states[:initially_infected] = STATE_INFECTED
    i_hist = []
    for step in range(steps):
        i_hist.append(np.sum(states == STATE_INFECTED))
        new_states = states.copy()
        for p in range(population):
            if states[p] == STATE_SUSCEPTIBLE:
                if is_infected_nearby_distance(p, positions, states, radius) \
                   and rng.random() < infection_prob:
                    new_states[p] = STATE_INFECTED
            elif states[p] == STATE_INFECTED:
                if rng.random() < recovery_prob:
                    new_states[p] = STATE_RECOVERED
        states = new_states
        for p in range(population):
            x, y = update_position(int(positions[p, 0]), int(positions[p, 1]), area, rng)
            positions[p] = [x, y]
    return np.array(i_hist)

fig, ax = plt.subplots(figsize=(10, 5))
for r in [0.5, 1.0, 2.0, 4.0]:
    i_curve = simulate_abm_sir_distance(radius=r, seed=42)
    ax.plot(i_curve, lw=2, label=f'radius = {r}')
ax.set_xlabel('Time step')
ax.set_ylabel('Infected agents')
ax.set_title('Effect of Infection Radius', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('Larger infection radius → faster spread, similar to increasing beta.')

---
## 8d · Vaccination Experiment

Let's pre-vaccinate some agents (set them to recovered state) and see how this affects the epidemic.

In [ ]:
def simulate_abm_sir_vaccinated(vaccinated_fraction=0.0, **kwargs):
    """Run the ABM with a fraction of agents pre-vaccinated (state = recovered)."""
    seed = kwargs.get('seed', 1)
    population = kwargs.get('population', 200)
    area = kwargs.get('area', 40)
    initially_infected = kwargs.get('initially_infected', 5)
    steps = kwargs.get('steps', 150)
    infection_probability = kwargs.get('infection_probability', 0.25)
    recovery_probability = kwargs.get('recovery_probability', 0.03)
    
    rng = random.Random(seed)
    positions = np.array([[rng.randint(0, area), rng.randint(0, area)]
                          for _ in range(population)], dtype=int)
    states = np.full(population, STATE_SUSCEPTIBLE, dtype=int)
    states[:initially_infected] = STATE_INFECTED
    
    # Vaccinate a fraction of the susceptible population
    n_vacc = int(vaccinated_fraction * (population - initially_infected))
    candidates = list(range(initially_infected, population))
    rng.shuffle(candidates)
    for idx in candidates[:n_vacc]:
        states[idx] = STATE_RECOVERED
    
    s_count = np.zeros(steps, dtype=int)
    i_count = np.zeros(steps, dtype=int)
    r_count = np.zeros(steps, dtype=int)
    
    for step in range(steps):
        s_count[step] = np.sum(states == STATE_SUSCEPTIBLE)
        i_count[step] = np.sum(states == STATE_INFECTED)
        r_count[step] = np.sum(states == STATE_RECOVERED)
        
        new_states = states.copy()
        for p in range(population):
            if states[p] == STATE_SUSCEPTIBLE:
                nearby = is_infected_nearby(p, positions, states)
                new_states[p] = update_health_state(
                    states[p], nearby, infection_probability, recovery_probability, rng=rng)
            elif states[p] == STATE_INFECTED:
                new_states[p] = update_health_state(
                    states[p], False, infection_probability, recovery_probability, rng=rng)
        states = new_states
        
        for p in range(population):
            x, y = update_position(int(positions[p, 0]), int(positions[p, 1]), area, rng=rng)
            positions[p] = [x, y]
    
    return {'susceptible': s_count, 'infected': i_count, 'recovered': r_count}

# Run with different vaccination levels
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 5))

for vf, c in zip([0.0, 0.2, 0.4, 0.6, 0.8], colors):
    run = simulate_abm_sir_vaccinated(vaccinated_fraction=vf, seed=42, steps=150)
    ax.plot(run['infected'], color=c, lw=2, label=f'{int(vf*100)}% vaccinated')

ax.set_xlabel('Time step')
ax.set_ylabel('Infected agents')
ax.set_title('Effect of Vaccination Coverage on Epidemic', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('Higher vaccination coverage flattens and suppresses the epidemic curve.')

## 8 · Explore randomness
Unlike the equation-based model, the agent-based model is stochastic. Two runs with the same parameters can produce different outcomes.


In [ ]:
seeds = [1, 2, 3, 4]
fig, ax = plt.subplots(figsize=(10, 5))

for seed in seeds:
    run = simulate_agent_sir(seed=seed, steps=120, population=200, initially_infected=5)
    ax.plot(run['infected'], linewidth=2, label=f'seed={seed}')

ax.set_title('Different runs of the same agent-based model')
ax.set_xlabel('Step')
ax.set_ylabel('Infected agents')
ax.legend()
plt.tight_layout()
plt.show()


## 9 · What to try next
Suggested experiments for your own exploration:

- Increase `infection_probability` and compare the peak number of infected agents
- Increase `recovery_probability` and see whether the epidemic dies out faster
- Change the `area` while keeping the population fixed and observe the effect of crowding
- Change the number of initially infected agents
- Replace the "same square" infection rule with a distance-based rule

### Questions to answer
1. Which parameter changes the spread most strongly?
2. Why do different seeds produce different curves?
3. In what ways is this model more realistic than the equation-based SIR model?
4. In what ways is it still simplified?


---
## 10 · Independent work and mini-project ideas
Choose one or more of these projects.

### Project A · Add an incubation state
Turn the model into an agent-based SEIR model by adding an exposed state.

### Project B · Add social distancing
Reduce movement or reduce contact probability after the infected count crosses a threshold.

### Project C · Add vaccination
Give some agents immunity at the beginning of the simulation.

### Project D · Add heterogeneity
Give different agents different movement speeds or different recovery probabilities.

### Project E · Compare ABM with the equation-based SIR model
Run both models with similar parameters and compare the epidemic curves.

### Final reflection
After finishing, describe:
- which features of real epidemics are naturally captured by agents,
- which assumptions are still unrealistic,
- what data you would need to make the model more realistic.
